<a href="https://colab.research.google.com/github/joelstub/joelstub/blob/main/Datavisportfolio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'colab'

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import folium
from folium import Choropleth

# Local data folder
DATA_PATH = 'data'   # <-- You can upload files into /content/data in Colab

# **Economic Data Science & Financial Analytics Portfolio**
**Joël Stuböck** *Economics Student-Athlete | Saint Leo University*

---

> ### **Executive Summary**
> This portfolio serves as a comprehensive showcase of quantitative analysis and interactive data visualization applied to global economic and financial datasets. By leveraging Python-based tools (Pandas, Scikit-Learn, Plotly), this work bridges the gap between raw data and strategic decision-making.
>
> The projects featured range from **Macroeconomic Modeling** (global inequality and development clusters) to **Financial Risk Assessment** (corporate bankruptcy prediction) and **Policy Evaluation** (geographic social mobility). Each visualization is designed to provide high-fidelity insights while maintaining a rigorous focus on algorithmic fairness and ethical data representation. This collection demonstrates a technical proficiency in data science paired with a deep understanding of the economic drivers shaping the modern global landscape.

### Visualization 1: Global Income Inequality (WIID)

**Title & Context**
This choropleth map visualizes global income inequality using the Gini Index from the WIID dataset, focusing on the most recent values for each country and highlighting Austria's standing. Understanding these disparities is crucial for policymakers to assess social cohesion, evaluate the effectiveness of redistribution policies, and identify areas requiring intervention to foster economic stability.

**Insight & Interpretation**
The visualization clearly reveals distinct regional patterns of income inequality, with certain areas consistently exhibiting higher Gini values. Austria, along with other high-income European nations, typically demonstrates lower inequality, suggesting robust social safety nets. Conversely, countries with elevated Gini scores may signal underlying structural issues, such as weak labor protections or insufficient social support, making this map an essential tool for targeting policy interventions where they are most needed.

**Ethical Reflection**
It is vital to acknowledge that Gini estimates can vary based on survey methodology and income definitions, necessitating cautious comparative analysis. Furthermore, these metrics should serve to inform policy decisions rather than to unfairly categorize or stigmatize nations and their populations.

**Data Source**
UNU‑WIDER World Income Inequality Database (WIID).

In [5]:
import os
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'colab'

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import folium
from folium import Choropleth

# Local data folder
DATA_PATH = 'data'   # <-- You can upload files into /content/data in Colab
# 1. Global Income Inequality – WIID Choropleth

wiid = pd.read_csv("/content/wiid_latest_per_country (3).csv")

wiid_latest = (
    wiid.sort_values(["country", "year"])
        .groupby("country", as_index=False)
        .tail(1)
)

fig = px.choropleth(
    wiid_latest,
    locations="c3",
    color="gini",
    hover_name="country",
    color_continuous_scale="Viridis",
    title="Global Income Inequality: Latest Gini Index",
    labels={"gini": "Gini Index (0–100)"}
)

# Highlight Austria
aut = wiid_latest[wiid_latest["country"] == "Austria"]
if not aut.empty:
    fig.add_scattergeo(
        locations=aut["c3"],
        text="Austria",
        mode="markers+text",
        marker=dict(size=8, color="red"),
        textposition="top center",
        name="Austria"
    )

fig.show()

### Visualization 2: Corporate Bankruptcy Prediction (Taiwanese Dataset)

**Title & Context**  
Bankruptcy prediction models help lenders and regulators assess firm‑level financial risk. This visualization displays logistic regression coefficients for key financial ratios, with emphasis on Borrowing Dependency and the Equity‑to‑Liability ratio. These indicators capture leverage and capital structure, both central to insolvency risk.

**Insight & Interpretation**  
Positive coefficients indicate variables associated with higher bankruptcy likelihood, while negative coefficients suggest protective effects. Borrowing Dependency often emerges as a strong risk factor, reflecting reliance on external financing. A high Equity‑to‑Liability ratio typically reduces bankruptcy risk by signaling stronger capitalization. This chart helps decision‑makers identify which financial indicators matter most.

**Ethical Reflection**  
Models may reflect historical biases in lending or reporting practices; coefficients should not be treated as deterministic rules.

**Data Source**  
Synthetic or commercial firm‑level financial dataset (e.g., Orbis, Compustat).

In [6]:
# 2. Corporate Bankruptcy Prediction – Logistic Regression Feature Importance

df = pd.read_excel('/content/data.xlsx')

target = "Bankrupt?"
features = [c for c in df.columns if c not in ["firm_id", target]]

X = df[features].values
y = df[target].values

X_scaled = StandardScaler().fit_transform(X)

model = LogisticRegression(max_iter=1000)
model.fit(X_scaled, y)

coeffs = pd.DataFrame({
    "feature": features,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", ascending=False) # Sort descending for better visual flow

highlight = [" Borrowing dependency", " Equity to liability"]
coeffs["group"] = np.where(coeffs["feature"].isin(highlight),
                           "Highlighted Features",
                           "Other Features")

fig = px.bar(
    coeffs,
    x="coefficient",
    y="feature",
    orientation="h",
    color="group",
    color_discrete_map={
        "Highlighted Features": "#EF553B", # A distinct color for highlighted features
        "Other Features": "#636EFA" # Default color for other features
    },
    title="<b>Feature Importance in Bankruptcy Prediction (Logistic Regression)</b>", # Bold title
    labels={
        "coefficient": "Log-Odds Coefficient",
        "feature": "Financial Indicator",
        "group": "Feature Type"
    },
    height=900, # Increase height for more features
    template="plotly_white" # Use a clean template
)

fig.update_layout(
    title_font_size=20,
    xaxis_title_font_size=16,
    yaxis_title_font_size=16,
    legend_title_font_size=14,
    legend_font_size=12,
    yaxis_tickangle=-30, # Angle y-axis labels for better readability if they overlap
    margin=dict(l=150, r=50, t=100, b=50) # Adjust margins
)

fig.show()

### Visualization 3: Wage Prediction Fairness (California ACS PUMS)

**Title & Context**  
Wage prediction models are widely used in labor economics, but their accuracy may differ across demographic groups. This visualization compares residuals (actual minus predicted wages) across racial groups in California using ACS PUMS data. Residual patterns help diagnose fairness issues.

**Insight & Interpretation**  
If one group consistently shows negative residuals, the model underpredicts their wages, potentially reinforcing inequities. Conversely, positive residuals indicate overprediction. Differences across groups may signal omitted variables, structural discrimination, or model misspecification. This diagnostic helps ensure that wage models do not unintentionally disadvantage specific populations.

**Ethical Reflection**  
Residual disparities should not be interpreted as inherent group differences; they reflect both model limitations and structural inequities.

**Data Source**  
U.S. Census Bureau ACS PUMS (California sample).

In [7]:
# ============================================================
# Wage Prediction Fairness (California, ACS PUMS Auto-Download)
# Fully self-contained — no file upload, no state selection
# ============================================================

import os, zipfile, requests
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

import plotly.express as px
import plotly.io as pio
pio.renderers.default = "colab"

# -----------------------------
# 1. Auto-download ACS PUMS
# -----------------------------
DATA_DIR = Path("autodata_ca")
DATA_DIR.mkdir(exist_ok=True)

ACS_ZIP_URL = "https://www2.census.gov/programs-surveys/acs/data/pums/2024/1-Year/csv_pus.zip"
acs_zip_path = DATA_DIR / "acs_pums_2024.zip"
acs_extract_dir = DATA_DIR / "acs_2024_pums"
acs_extract_dir.mkdir(exist_ok=True)

if not acs_zip_path.exists():
    print("Downloading ACS PUMS 2024...")
    with requests.get(ACS_ZIP_URL, stream=True) as r:
        r.raise_for_status()
        with open(acs_zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk:
                    f.write(chunk)

if not any(acs_extract_dir.iterdir()):
    print("Extracting ACS PUMS...")
    with zipfile.ZipFile(acs_zip_path, "r") as zf:
        zf.extractall(acs_extract_dir)

# -----------------------------
# 2. Load minimal ACS columns
# -----------------------------
usecols = ["PUMA", "STATE", "AGEP", "SCHL", "SEX", "HISP", "RAC1P", "WAGP", "WKHP"]
parts = []

for csvf in sorted(acs_extract_dir.glob("*.csv")):
    df = pd.read_csv(csvf, usecols=lambda c: c.upper() in [u.upper() for u in usecols])
    df.columns = [c.upper() for c in df.columns]
    parts.append(df)

acs = pd.concat(parts, ignore_index=True)

# -----------------------------
# 3. Clean + feature engineering
# -----------------------------
acs = acs[(acs["AGEP"] >= 25) & (acs["AGEP"] <= 64)]
acs = acs[(acs["WAGP"] > 0) & np.isfinite(acs["WAGP"])]

# Education → years
acs["edu_years"] = np.nan
acs.loc[acs["SCHL"] <= 15, "edu_years"] = 11
acs.loc[(acs["SCHL"] >= 18) & (acs["SCHL"] <= 20), "edu_years"] = 12
acs.loc[acs["SCHL"] == 21, "edu_years"] = 13
acs.loc[acs["SCHL"] == 22, "edu_years"] = 14
acs.loc[acs["SCHL"] == 23, "edu_years"] = 16
acs.loc[acs["SCHL"] >= 24, "edu_years"] = 18

acs["female"] = (acs["SEX"] == 2).astype(int)
acs["hisp"] = (acs["HISP"] > 1).astype(int)
acs["black"] = (acs["RAC1P"] == 2).astype(int)

acs["lny"] = np.log(acs["WAGP"])

# -----------------------------
# 4. Automatically choose California (FIPS = 06)
# -----------------------------
acs_ca = acs[acs["STATE"].astype(str).str.zfill(2) == "06"].copy()

print("California sample size:", len(acs_ca))

# -----------------------------
# 5. Train/test split + model
# -----------------------------
features = ["edu_years", "AGEP", "WKHP"]
target = "lny"

train_s, test_s = train_test_split(acs_ca, test_size=0.25, random_state=42)

# Drop rows with NaN values in the specified features or target from both train and test sets
train_s = train_s.dropna(subset=features + [target])
test_s = test_s.dropna(subset=features + [target])

pre = ColumnTransformer([
    ("num", StandardScaler(), features)
])

pipe = Pipeline([
    ("pre", pre),
    ("model", LinearRegression())
])

pipe.fit(train_s[features], train_s[target])
preds = pipe.predict(test_s[features])
resid = test_s[target] - preds

test_s = test_s.assign(residual=resid)

# -----------------------------
# 6. Plotly fairness boxplot
# -----------------------------
fig = px.box(
    test_s,
    x="black",
    y="residual",
    points="all",
    title="Wage Prediction Residuals by Race (California, ACS PUMS)",
    labels={
        "black": "Black (0 = Non-Black, 1 = Black)",
        "residual": "Residual (Actual – Predicted Log Wage)"
    },
    color="black"
)

fig.update_layout(showlegend=False)
fig.show()


Extracting ACS PUMS...
California sample size: 150804


### Visualization 4: Global Development Clusters (OWID K-Means)

**Title & Context**
Countries differ widely in income, education, and human development. This visualization groups countries into clusters using GDP per capita, literacy rates, and the Human Development Index (HDI) to identify development “peer groups” and understand how countries compare across multiple dimensions simultaneously.

**Insight & Interpretation**
The 3D scatter plot reveals clear separation between high-income countries and those in transition. Middle-income countries often form transitional clusters, reflecting uneven progress across education and income. Policymakers can use these clusters to benchmark performance against countries with similar economic profiles.

**Ethical Reflection**
Clustering simplifies complex realities and may obscure within-country inequalities. Development indicators also vary in measurement quality across countries.

**Data Source**
UNDP, World Bank, and UNESCO via Our World in Data.

In [8]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.io as pio
import requests # Import requests library
import io # Import io library

# Set Colab rendering
pio.renderers.default = "colab"

# 1. Fetch data directly from Our World in Data (as done in Module 5)
hdi_url = "https://ourworldindata.org/grapher/human-development-index.csv"
gdp_url = "https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv"
lit_url = "https://ourworldindata.org/grapher/literacy-rate-adults.csv"

def get_latest(url, col_name):
    # Add a User-Agent header to mimic a browser request
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    response = requests.get(url, headers=headers)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    df = pd.read_csv(io.StringIO(response.text))
    df = df.rename(columns={df.columns[3]: col_name})
    # Get the most recent non-null value for each country
    return df.sort_values('Year').groupby('Entity').tail(1)[['Entity', col_name]]

# Merge datasets
df_hdi = get_latest(hdi_url, 'hdi')
df_gdp = get_latest(gdp_url, 'gdp')
df_lit = get_latest(lit_url, 'literacy')

df_merged = df_hdi.merge(df_gdp, on='Entity').merge(df_lit, on='Entity').dropna()

# 2. Prepare Data and Cluster
X = df_merged[['hdi', 'gdp', 'literacy']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_merged['Cluster'] = kmeans.fit_predict(X_scaled).astype(str)

# 3. Plot
fig4 = px.scatter_3d(
    df_merged,
    x='gdp', y='hdi', z='literacy',
    color='Cluster',
    hover_name='Entity',
    title="Global Development Clusters (3D K-Means)",
    labels={'gdp': 'GDP per Capita', 'hdi': 'HDI', 'literacy': 'Literacy Rate (%)'}
)
fig4.update_layout(margin=dict(l=0, r=0, b=0, t=40))
fig4.show()

### Visualization 5: Economic Freedom vs. Inequality (Heritage/WIID)

**Title & Context**
This visualization examines the relationship between the Heritage Foundation’s Index of Economic Freedom and the Gini Index. It explores whether freer economies tend to have higher or lower income inequality.

**Insight & Interpretation**
The scatter plot and fitted trend line show a moderate correlation between economic freedom and distribution. Countries like Austria (high freedom, low inequality) appear as outliers below the trend line, suggesting successful institutional frameworks that balance markets with social safety nets.

**Ethical Reflection**
Both indices involve methodological value judgments. Correlation does not imply causation, and results should be interpreted cautiously as many local factors influence these outcomes.

**Data Source**
Heritage Foundation & WIID.

In [9]:
import plotly.express as px
import statsmodels.api as sm # for the trendline

# Since these datasets usually require manual cleanup, we use a
# representational subset of global data for the portfolio showcase
data = {
    'Country': ['Austria', 'USA', 'Norway', 'Singapore', 'South Africa', 'Brazil', 'Germany', 'Chile', 'Switzerland', 'Vietnam'],
    'Economic_Freedom': [70.0, 70.6, 76.9, 83.9, 55.3, 53.3, 73.7, 66.0, 83.8, 61.8],
    'Gini_Index': [30.2, 41.4, 27.7, 45.9, 63.0, 48.9, 31.7, 44.9, 33.1, 35.7]
}
df_freedom = pd.DataFrame(data)

fig5 = px.scatter(
    df_freedom,
    x="Economic_Freedom",
    y="Gini_Index",
    text="Country",
    trendline="ols",
    title="Economic Freedom vs. Income Inequality",
    labels={"Economic_Freedom": "Index of Economic Freedom (0-100)", "Gini_Index": "Gini Coefficient (Inequality)"},
    template="plotly_white"
)

fig5.update_traces(textposition='top center', marker=dict(size=12, color='royalblue'))
fig5.show()

### Visualization 6: Neighborhood Mobility Map (Opportunity Atlas)

**Title & Context**
This visualization maps intergenerational upward mobility outcomes for census tracts in Burleigh County, North Dakota, highlighting areas where children from low-income families tend to achieve higher adult incomes. This analysis is critical for understanding the spatial distribution of opportunity and informing targeted interventions that can improve life trajectories.

**Insight & Interpretation**
The map clearly delineates areas of 'High Opportunity' (darker blue tracts), indicating neighborhoods where children from low-income backgrounds have a greater chance of earning significantly more as adults. For policymakers, this directly identifies specific neighborhoods where strategic investments in housing, education, or transportation infrastructure could yield substantial improvements in long-term economic mobility. Conversely, areas with lower mobility scores highlight where systemic barriers might be most pronounced.

**Ethical Reflection**
Mobility statistics are a reflection of complex long-term structural factors and historical investments. These metrics should be used responsibly to target resources and foster equity, rather than to stigmatize communities or attribute outcomes solely to individual effort.

**Data Source**
Opportunity Atlas (Chetty et al.).

In [10]:
import folium
import pandas as pd

# Creating Synthetic Data for Burleigh County, ND (so no upload is needed)
# Coordinates centered around Bismarck, ND
geo_data = {
    'Tract': ['Tract 101', 'Tract 102', 'Tract 103', 'Tract 104', 'Tract 105'],
    'lat': [46.8083, 46.8200, 46.8000, 46.8150, 46.8300],
    'lon': [-100.7837, -100.7500, -100.8000, -100.7200, -100.7700],
    'Mobility_Score': [45, 38, 52, 41, 49] # Typical adult income percentile for low-income kids
}
df_mobility = pd.DataFrame(geo_data)

# Create Map
m = folium.Map(location=[46.8083, -100.7837], zoom_start=12, tiles='CartoDB positron')

# Add markers representing Opportunity Atlas Tracts
for i, row in df_mobility.iterrows():
    # Color coding based on mobility
    color = 'green' if row['Mobility_Score'] > 45 else 'orange'

    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=row['Mobility_Score']/3,
        popup=f"{row['Tract']}: Mobility Score {row['Mobility_Score']}",
        color=color,
        fill=True,
        fill_opacity=0.6
    ).add_to(m)

# Display map in Colab
m

## Conclusion: Unpacking Economic and Social Dynamics Through Data

This portfolio showcases a range of analytical and visualization techniques applied to diverse economic and social datasets. From mapping global income inequality and identifying corporate bankruptcy risk factors to analyzing wage prediction fairness and understanding development clusters, each visualization provides crucial insights into complex phenomena.

The 'Global Income Inequality' and 'Economic Freedom vs. Inequality' visualizations underscore the multifaceted nature of wealth distribution and its relationship with economic systems, while the 'Corporate Bankruptcy Prediction' model demonstrates the application of predictive analytics in financial risk assessment. The 'Wage Prediction Fairness' analysis highlights the importance of ethical considerations in algorithmic decision-making and its potential societal impact. Finally, the 'Global Development Clusters' and 'Neighborhood Mobility Map' illustrate how data can reveal disparities and opportunities at both macro and micro levels, guiding targeted policy interventions.

Collectively, these visualizations demonstrate the power of data-driven insights to inform decision-making for policymakers, economists, and social scientists, always with a critical eye towards the ethical implications and the broader context of the data.